## 1. Dataset Overview & Class Distributions

We first inspect the basic structure of our final modelling datasets.

We report:
- the number of user-target pairs,
- the number of unique users,
- and the overall distribution of stance labels.

For this section, we only load the columns required for the analysis (`UserId`, `TargetEntity`, and `StanceLabel`) instead of the full posting histories.

In [ ]:
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq

DATA_DIR = Path("../data/preprocessed")

TRAIN_PATH = DATA_DIR / "train.parquet"
VALIDATION_PATH = DATA_DIR / "validation.parquet"
HUMAN_TEST_PATH = DATA_DIR / "human_test.parquet"

analysis_columns = [
    "UserId",
    "TargetEntity",
    "StanceLabel",
]

train_df = pq.read_table(
    TRAIN_PATH,
    columns=analysis_columns
).to_pandas()

validation_df = pq.read_table(
    VALIDATION_PATH,
    columns=analysis_columns
).to_pandas()

human_test_df = pq.read_table(
    HUMAN_TEST_PATH,
    columns=analysis_columns
).to_pandas()


print("Train rows:", len(train_df))
print("Validation rows:", len(validation_df))
print("Human test rows:", len(human_test_df))

Train rows: 12834
Validation rows: 3210
Human test rows: 890


In [4]:
# Dataset overview

overview = pd.DataFrame({
    "Dataset": [
        "Train",
        "Validation",
        "Human test",
    ],
    "User-target pairs": [
        len(train_df),
        len(validation_df),
        len(human_test_df),
    ],
    "Unique users": [
        train_df["UserId"].nunique(),
        validation_df["UserId"].nunique(),
        human_test_df["UserId"].nunique(),
    ],
})

display(overview)


# Overall stance-class distributions

datasets = {
    "Train": train_df,
    "Validation": validation_df,
    "Human test": human_test_df,
}

for name, df in datasets.items():

    counts = df["StanceLabel"].value_counts()

    percentages = (
        df["StanceLabel"]
        .value_counts(normalize=True)
        .mul(100)
        .round(1)
    )

    distribution = pd.DataFrame({
        "Count": counts,
        "Percentage": percentages,
    })

    print(f"\n{name}")
    display(distribution)

,Dataset,User-target pairs,Unique users
0,Train,12834,6417
1,Validation,3210,1605
2,Human test,890,445



Train


,Count,Percentage
StanceLabel,,
Against,6255,48.7
Neither,4147,32.3
Favor,2432,18.9



Validation


,Count,Percentage
StanceLabel,,
Against,1577,49.1
Neither,994,31.0
Favor,639,19.9



Human test


,Count,Percentage
StanceLabel,,
Against,440,49.4
Favor,235,26.4
Neither,215,24.2


---
## 2. Trump vs. Harris

Next, we inspect the stance-label distributions separately for Trump and Harris.

This helps us determine whether the overall class imbalance is mainly driven by one of the two political targets and whether the target-specific distributions differ between the train, validation, and human-test sets.

In [5]:
# Define a consistent label order for easier comparison
label_order = ["Against", "Neither", "Favor"]

for name, df in datasets.items():

    print(f"\n{name}")

    for target in ["Trump", "Harris"]:

        target_df = df[df["TargetEntity"] == target]

        counts = (
            target_df["StanceLabel"]
            .value_counts()
            .reindex(label_order, fill_value=0)
        )

        percentages = (
            target_df["StanceLabel"]
            .value_counts(normalize=True)
            .mul(100)
            .reindex(label_order, fill_value=0)
            .round(1)
        )

        distribution = pd.DataFrame({
            "Count": counts,
            "Percentage": percentages,
        })

        print(f"\n{target}")
        display(distribution)


Train

Trump


,Count,Percentage
StanceLabel,,
Against,5395,84.1
Neither,747,11.6
Favor,275,4.3



Harris


,Count,Percentage
StanceLabel,,
Against,860,13.4
Neither,3400,53.0
Favor,2157,33.6



Validation

Trump


...

,Count,Percentage
StanceLabel,,
Against,213,13.3
Neither,823,51.3
Favor,569,35.5



Human test

Trump


,Count,Percentage
StanceLabel,,
Against,390,87.6
Neither,39,8.8
Favor,16,3.6



Harris


,Count,Percentage
StanceLabel,,
Against,50,11.2
Neither,176,39.6
Favor,219,49.2


In [22]:
comparison_tables = []

for name, df in datasets.items():

    target_distribution = pd.crosstab(
        df["TargetEntity"],
        df["StanceLabel"],
        normalize="index"
    ).mul(100)

    target_distribution = target_distribution.reindex(
        columns=label_order,
        fill_value=0
    )

    target_distribution["Dataset"] = name

    comparison_tables.append(
        target_distribution.reset_index()
    )

target_comparison = pd.concat(
    comparison_tables,
    ignore_index=True
)

# Arrange columns and remove the crosstab column-axis label
target_comparison = target_comparison[
    ["Dataset", "TargetEntity", "Against", "Neither", "Favor"]
]

target_comparison.columns.name = None

target_comparison[["Against", "Neither", "Favor"]] = (
    target_comparison[["Against", "Neither", "Favor"]]
    .round(1)
)

# Hide the unnecessary numeric DataFrame index
display(
    target_comparison.style
    .format({
        "Against": "{:.1f}",
        "Neither": "{:.1f}",
        "Favor": "{:.1f}",
    })
    .hide(axis="index")
)

Dataset,TargetEntity,Against,Neither,Favor
Train,Harris,13.4,53.0,33.6
Train,Trump,84.1,11.6,4.3
Validation,Harris,13.3,51.3,35.5
Validation,Trump,85.0,10.7,4.4
Human test,Harris,11.2,39.6,49.2
Human test,Trump,87.6,8.8,3.6


---
## 3. Posting-History Length / Available Posts

We now examine how much posting history is available for each user.

Each user has a posting history containing up to 1,000 posts. The number of available posts is important because these histories form the evidence pool from which posts will later be retrieved for stance prediction.

We examine:
- the distribution of posting-history lengths,
- summary statistics such as mean, median, minimum, and maximum,
- and how many users have relatively short histories.

In [8]:
history_path = DATA_DIR / "user_posting_histories.parquet"

history_table = pq.read_table(
    history_path,
    columns=["UserId", "PostingHistory"]
)

history_lengths = pd.DataFrame({
    "UserId": history_table["UserId"].to_pylist(),
    "NumPosts": [
        len(posts) if posts is not None else 0
        for posts in history_table["PostingHistory"].to_pylist()
    ]
})

display(history_lengths.head())

,UserId,NumPosts
0,9,1000
1,10,1000
2,49,27
3,105,95
4,144,130


In [9]:
# Summary statistics
summary = history_lengths["NumPosts"].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
).to_frame("NumPosts")

display(summary.round(1))


# Check how many users have fewer than different numbers of available posts
k_values = [1, 2, 5, 10, 20, 50, 100, 500, 1000]

availability = pd.DataFrame({
    "k": k_values,
    "Users with fewer than k posts": [
        (history_lengths["NumPosts"] < k).sum()
        for k in k_values
    ],
})

availability["Percentage"] = (
    availability["Users with fewer than k posts"]
    / len(history_lengths)
    * 100
).round(1)

display(availability)

print(
    "Users with exactly 1,000 posts:",
    (history_lengths["NumPosts"] == 1000).sum()
)

,NumPosts
count,8467.0
mean,336.0
std,351.8
min,1.0
25%,69.0
50%,166.0
75%,519.5
90%,1000.0
95%,1000.0
99%,1000.0


,k,Users with fewer than k posts,Percentage
0,1,0,0.0
1,2,29,0.3
2,5,96,1.1
3,10,199,2.4
4,20,467,5.5
5,50,1476,17.4
6,100,3014,35.6
7,500,6289,74.3
8,1000,7119,84.1


Users with exactly 1,000 posts: 1348


---
## 4. Candidate-Name Frequencies

We next examine how often the two target candidates are explicitly mentioned in users' posting histories.

For each user, we count the number of posts containing:
- `Trump`
- `Harris`

We count posts rather than individual word occurrences, so a post is counted at most once per candidate.

This provides a first indication of how much direct candidate-related evidence is available and whether explicit candidate names could potentially act as simple lexical shortcuts.

In [11]:
import re


# Match candidate surnames as complete words, case-insensitive
trump_pattern = re.compile(r"\btrump\b", re.IGNORECASE)
harris_pattern = re.compile(r"\bharris\b", re.IGNORECASE)

candidate_mentions = []

# Count how many posts per user mention each candidate
for user_id, posts in zip(
    history_table["UserId"].to_pylist(),
    history_table["PostingHistory"].to_pylist()
):
    trump_posts = 0
    harris_posts = 0

    for post in posts:
        content = post.get("Content") or ""

        if trump_pattern.search(content):
            trump_posts += 1

        if harris_pattern.search(content):
            harris_posts += 1

    candidate_mentions.append({
        "UserId": user_id,
        "NumPosts": len(posts),
        "TrumpPosts": trump_posts,
        "HarrisPosts": harris_posts,
    })

candidate_mentions = pd.DataFrame(candidate_mentions)

display(candidate_mentions.head())

,UserId,NumPosts,TrumpPosts,HarrisPosts
0,9,1000,69,15
1,10,1000,128,65
2,49,27,9,1
3,105,95,6,3
4,144,130,32,4


In [12]:
total_posts = candidate_mentions["NumPosts"].sum()
total_users = len(candidate_mentions)

# Aggregate post-level and user-level candidate mentions
candidate_summary = pd.DataFrame({
    "Candidate": ["Trump", "Harris"],
    "Posts mentioning candidate": [
        candidate_mentions["TrumpPosts"].sum(),
        candidate_mentions["HarrisPosts"].sum(),
    ],
    "Users mentioning candidate": [
        (candidate_mentions["TrumpPosts"] > 0).sum(),
        (candidate_mentions["HarrisPosts"] > 0).sum(),
    ],
})

# Convert absolute counts into percentages for easier comparison
candidate_summary["Percentage of all posts"] = (
    candidate_summary["Posts mentioning candidate"]
    / total_posts
    * 100
).round(1)

candidate_summary["Percentage of users"] = (
    candidate_summary["Users mentioning candidate"]
    / total_users
    * 100
).round(1)

display(candidate_summary)

,Candidate,Posts mentioning candidate,Users mentioning candidate,Percentage of all posts,Percentage of users
0,Trump,382207,8188,13.4,96.7
1,Harris,63147,5731,2.2,67.7


---
## 5. Majority Baselines

Finally, we establish simple majority-class baselines against which later models can be compared.

We consider two baselines:

- **Global majority baseline:** always predicts the most frequent stance label in the training set.
- **Target-conditioned majority baseline:** predicts the most frequent training label separately for Trump and Harris.

The target-conditioned baseline is particularly relevant because the stance distributions differ strongly between the two candidates.

Both baselines are derived exclusively from the training data and evaluated on the validation and human-test sets using accuracy and macro-F1.

In [14]:
from sklearn.metrics import accuracy_score, f1_score


# Determine the overall majority class from the training set
global_majority = train_df["StanceLabel"].mode()[0]

# Determine the majority class separately for each target
target_majorities = (
    train_df.groupby("TargetEntity")["StanceLabel"]
    .agg(lambda x: x.mode()[0])
    .to_dict()
)

print("Global majority:", global_majority)
print("Target-conditioned majorities:", target_majorities)

Global majority: Against
Target-conditioned majorities: {'Harris': 'Neither', 'Trump': 'Against'}


In [23]:
import sys

# Add the project root so modules inside src can be imported
sys.path.append(str(Path("..").resolve()))

from src.utils import evaluate_predictions

baseline_results = []

for dataset_name, df in {
    "Validation": validation_df,
    "Human test": human_test_df,
}.items():

    # Global majority: predict the same class for every user-target pair
    global_predictions = [global_majority] * len(df)

    global_scores = evaluate_predictions(
        df["StanceLabel"],
        global_predictions,
        labels=label_order,
    )

    baseline_results.append({
        "Dataset": dataset_name,
        "Baseline": "Global majority",
        **global_scores,
    })

    # Target-conditioned majority: use a different majority class per candidate
    target_predictions = df["TargetEntity"].map(target_majorities)

    target_scores = evaluate_predictions(
        df["StanceLabel"],
        target_predictions,
        labels=label_order,
    )

    baseline_results.append({
        "Dataset": dataset_name,
        "Baseline": "Target-conditioned majority",
        **target_scores,
    })


baseline_results = pd.DataFrame(baseline_results)

baseline_results[["Accuracy", "Macro-F1"]] = (
    baseline_results[["Accuracy", "Macro-F1"]]
    .mul(100)
    .round(1)
)

display(baseline_results)

,Dataset,Baseline,Accuracy,Macro-F1
0,Validation,Global majority,49.1,22.0
1,Validation,Target-conditioned majority,68.1,49.7
2,Human test,Global majority,49.4,22.1
3,Human test,Target-conditioned majority,63.6,47.2
